In [14]:
"""
🏃 Endurance Sports RAG – Pure Python Agent

Architecture:
1. Query Rewriting (LLM → optimized search query)
2. Two-Stage Retrieval:
   - Stage 1: OpenAI embeddings + Chroma vector search
   - Stage 2: HuggingFace CrossEncoder reranking
3. Controlled Generation (LLM synthesis)
4. Gradio UI
"""

import os
import openai
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import CrossEncoder
import gradio as gr
from dotenv import load_dotenv
load_dotenv()  # loads variables from .env

# ─────────────────────────────────────────────────────────────
# 1. AUTHENTICATION
# ─────────────────────────────────────────────────────────────

OPENAI_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_KEY:
    raise ValueError("❌ OPENAI_API_KEY is missing. Set it before running.")

client = openai.OpenAI(api_key=OPENAI_KEY)
print("✅ OpenAI client initialized.")



✅ OpenAI client initialized.


In [15]:

# ─────────────────────────────────────────────────────────────
# 2. CONNECT TO VECTOR DATABASE
# ─────────────────────────────────────────────────────────────

DB_PATH = "./endurance_db"
COLLECTION_NAME = "endurance_knowledge_base"

print("🧠 Connecting to local Chroma database...")
chroma_client = chromadb.PersistentClient(path=DB_PATH)

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_KEY,
    model_name="text-embedding-3-small"
)

collection = chroma_client.get_collection(
    name=COLLECTION_NAME,
    embedding_function=openai_ef
)

print("⚖️ Loading HuggingFace reranker (BAAI/bge-reranker-base)...")
reranker = CrossEncoder("BAAI/bge-reranker-base")

print(f"✅ Connected. Database contains {collection.count()} chunks.")



🧠 Connecting to local Chroma database...
⚖️ Loading HuggingFace reranker (BAAI/bge-reranker-base)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1351.80it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Connected. Database contains 526 chunks.


In [16]:

# ─────────────────────────────────────────────────────────────
# 3. QUERY REWRITING
# ─────────────────────────────────────────────────────────────

def rewrite_query(user_message: str) -> str:
    """
    Converts conversational user message into optimized
    search query for vector retrieval.
    """

    prompt = f"""
You are an expert endurance sports data engineer.
Convert the user's conversational message into a highly specific,
clinical search query optimized for physiological metrics
(HR, Pace, Distance) and scientific sports abstracts.

User message: "{user_message}"

Output ONLY the rewritten search query.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
    )

    return response.choices[0].message.content.strip()




In [17]:
# ─────────────────────────────────────────────────────────────
# 4. TWO-STAGE RETRIEVAL
# ─────────────────────────────────────────────────────────────

def retrieve_and_rerank(search_query: str, top_k: int = 3) -> str:
    """
    Stage 1: Vector search (coarse retrieval)
    Stage 2: CrossEncoder reranking
    """

    # Stage 1
    results = collection.query(
        query_texts=[search_query],
        n_results=15
    )

    candidates = results["documents"][0]
    metadatas = results["metadatas"][0]

    if not candidates:
        return ""

    # Stage 2
    pairs = [[search_query, doc] for doc in candidates]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, candidates, metadatas),
        key=lambda x: x[0],
        reverse=True
    )

    context_blocks = []
    for score, doc, meta in ranked[:top_k]:
        context_blocks.append(
            f"[Source: {meta.get('source', 'Unknown')} | "
            f"Type: {meta.get('document_type', 'Unknown')}]\n{doc}"
        )

    return "\n\n---\n\n".join(context_blocks)



In [18]:

# ─────────────────────────────────────────────────────────────
# 5. FINAL GENERATION
# ─────────────────────────────────────────────────────────────

def generate_response(user_message, chat_history):
    """
    Full RAG pipeline:
    1. Rewrite
    2. Retrieve
    3. Generate controlled answer
    """

    # Rewrite
    search_query = rewrite_query(user_message)
    print(f"\n[DEBUG] Rewritten Query:\n{search_query}")

    # Retrieve
    context = retrieve_and_rerank(search_query)
    print(f"[DEBUG] Retrieved Context:\n{context}")

    system_prompt = f"""
You are an elite endurance sports coach and data analyst.
You MUST answer using ONLY the data in the provided context.
If the context does not contain the answer, state that
you do not have enough data.

CONTEXT:
{context}
"""

    messages = [{"role": "system", "content": system_prompt}]

    for past_user, past_bot in chat_history:
        messages.append({"role": "user", "content": past_user})
        messages.append({"role": "assistant", "content": past_bot})

    messages.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0.3,
    )

    return response.choices[0].message.content




In [ ]:
# ─────────────────────────────────────────────────────────────
# 6. LAUNCH UI
# ─────────────────────────────────────────────────────────────

def main():
    print("🚀 Launching Gradio interface...")

    demo = gr.ChatInterface(
        fn=generate_response,
        title="🏃 AI Endurance Coach (Pure Python RAG)",
        description=(
            "Ask about physiological metrics or endurance science. "
            "This system uses query rewriting + 2-stage retrieval."
        ),
        examples=[
            "I had a really tiring, long distance run recently. What does the data say?",
            "What scientific literature do you have on tapering?",
            "Based on my logs, what is my average heart rate on long runs?"
        ],
    )

    demo.launch(share=False)


if __name__ == "__main__":
    main()

🚀 Launching Gradio interface...
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.



[DEBUG] Rewritten Query:
"10km race physiological metrics heart rate pace distance performance analysis"
[DEBUG] Retrieved Context:
[Source: kaggle_cardio_activities | Type: workout_log]
On 2018-10-19 17:52:32, the athlete completed a running workout. They covered a distance of 10.29 km in a total duration of 59:18. The physiological stress was measured with an average heart rate of 155.0 bpm.

---

[Source: kaggle_cardio_activities | Type: workout_log]
On 2018-11-11 14:05:12, the athlete completed a running workout. They covered a distance of 10.44 km in a total duration of 58:40. The physiological stress was measured with an average heart rate of 159.0 bpm.

---

[Source: kaggle_cardio_activities | Type: workout_log]
On 2018-08-05 10:17:10, the athlete completed a running workout. They covered a distance of 18.7 km in a total duration of 1:49:19. The physiological stress was measured with an average heart rate of 150.0 bpm.
